# RipEye damage detection training — Google Colab

This is the Colab version of `notebooks/train_damage_detection.ipynb`.

It trains the YOLO damage detector and produces the weights file used by the local app:

`runs/detect/ripeye/weights/best.pt`

Recommended Colab runtime: **T4 GPU** or better.

## 0. Runtime check

In Colab, go to **Runtime → Change runtime type → Hardware accelerator → GPU** before running training.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training will work, but it will be much slower.")

## 1. Install dependencies

This installs the training stack inside the Colab runtime.

In [ ]:
%pip install -q ultralytics pandas pyyaml matplotlib seaborn scikit-learn roboflow python-dotenv

## 2. Get the RipEye code

If you uploaded this notebook alone, run this cell to clone the repo. If you already uploaded the whole repo to Colab/Drive, set `ROOT` to that folder instead.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "https://github.com/ElianaLim/RipEye.git"
WORKSPACE = Path("/content")
ROOT = WORKSPACE / "RipEye"

if not ROOT.exists():
    !git clone {REPO_URL} {ROOT}
else:
    print(f"Using existing repo at {ROOT}")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

## 3. Download / prepare the dataset

This project expects a YOLO dataset under `data/`.

The helper script downloads Roboflow versions 1 and 2, merges them, and generates `data/severity_labels.csv`.

You need a Roboflow API key for this cell. In Colab, you can either:

- paste it when prompted, or
- store it in Colab secrets as `ROBOFLOW_API_KEY`.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    secret = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    secret = None

if secret:
    os.environ["ROBOFLOW_API_KEY"] = secret
elif not os.environ.get("ROBOFLOW_API_KEY"):
    os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

!python scripts/fetch_data.py

## 4. Configure paths and training settings

You can lower `EPOCHS_MAX` for a quick smoke test, for example `EPOCHS_MAX = 3`.
For real training, keep the default and let early stopping decide when to stop.

In [ ]:
from pathlib import Path
import yaml

DATASET_DIR = ROOT / "data"
DATA_YAML = DATASET_DIR / "data.yaml"
RUN_NAME = "ripeye"
WEIGHTS = ROOT / "runs/detect" / RUN_NAME / "weights" / "best.pt"

assert DATA_YAML.exists(), f"Missing {DATA_YAML}"

EPOCHS_MAX = 100
PATIENCE = 15
IMG_SIZE = 640
MODEL_CKPT = "yolov8s.pt"
PREDICT_CONF = 0.15
EVAL_SPLIT = 0.2
RANDOM_STATE = 42

print(DATA_YAML.read_text())
print("Weights will be saved to:", WEIGHTS)

## 5. Explore dataset

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

meta = yaml.safe_load(DATA_YAML.read_text())
class_names = meta.get("names", [])
if isinstance(class_names, dict):
    class_names = [class_names[k] for k in sorted(class_names, key=int)]

print("Classes:", class_names)
for split in ("train", "valid", "test"):
    img_dir = DATASET_DIR / split / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*")))
        print(f"  {split}: {n} images")

box_counts = Counter()
labels_dir = DATASET_DIR / "train" / "labels"
for lf in labels_dir.glob("*.txt"):
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            box_counts[int(line.split()[0])] += 1

print("\nBoxes per class (train):")
for cid, count in sorted(box_counts.items()):
    name = class_names[cid] if cid < len(class_names) else str(cid)
    print(f"  {name}: {count}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([class_names[i] for i in sorted(box_counts)], [box_counts[i] for i in sorted(box_counts)])
ax.set_title("Ground-truth boxes per class")
ax.set_ylabel("count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Build severity labels

Severity is derived from box area:

- `none`: no damage box
- `minor`: damage area is greater than 0 and up to 30% of package area
- `severe`: severe class detected or damage area is above 30%

In [ ]:
import pandas as pd
from ripeye.severity import SeverityConfig, build_severity_csv

CFG = SeverityConfig(severe_ratio=0.30)
csv_path = DATASET_DIR / "severity_labels.csv"
build_severity_csv(DATASET_DIR, csv_path, cfg=CFG)
df = pd.read_csv(csv_path)
print(df["severity"].value_counts())
df.head(8)

In [ ]:
df["severity"].value_counts().plot.bar(
    title="GT severity (none / minor / severe)", color=["#94a3b8", "#f59e0b", "#ef4444"]
)
plt.ylabel("images")
plt.tight_layout()
plt.show()

df["damage_area_ratio"].hist(bins=30, edgecolor="white")
plt.title("GT damage area ratio distribution")
plt.xlabel("damage_area / package_area")
plt.tight_layout()
plt.show()

## 7. Train YOLO

This is the slow cell. On Colab, `DEVICE` should usually be `0`, meaning the first CUDA GPU.

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

model = YOLO(MODEL_CKPT)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS_MAX,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    cache=True,
    project=str(ROOT / "runs/detect"),
    name=RUN_NAME,
    exist_ok=True,
)
print("Weights:", WEIGHTS)
print("Exists:", WEIGHTS.exists())

## 8. Detection metrics

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

best = YOLO(str(WEIGHTS)) if WEIGHTS.exists() else model
det_metrics = best.val(data=str(DATA_YAML), device=DEVICE)
print(det_metrics)

for name in ("results.png", "confusion_matrix.png", "BoxF1_curve.png"):
    p = ROOT / "runs/detect" / RUN_NAME / name
    if p.exists():
        print(name)
        display(IPImage(filename=str(p)))

## 9. Severity accuracy

This evaluates the app-level label: `none`, `minor`, or `severe`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from ripeye.severity import load_id_to_name, severity_for_prediction

id_to_name = load_id_to_name(DATA_YAML)

valid_dir = DATASET_DIR / "valid" / "images"
if valid_dir.exists() and any(valid_dir.iterdir()):
    eval_df = df[df["split"] == "val"].copy()
    if eval_df.empty:
        eval_df = df[df["label_file"].str.startswith("valid/")].copy()
    img_roots = [valid_dir]
    print(f"Eval on Roboflow valid split: {len(eval_df)} images")
else:
    eval_df, _ = train_test_split(
        df,
        test_size=EVAL_SPLIT,
        random_state=RANDOM_STATE,
        stratify=df["severity"],
    )
    img_roots = [DATASET_DIR / "train" / "images"]
    print(f"Holdout eval: {len(eval_df)} images ({EVAL_SPLIT:.0%} of {len(df)})")

y_true, y_pred = [], []

for _, row in eval_df.iterrows():
    stem = row["image_stem"]
    img_path = None
    for root in img_roots:
        img_path = next(root.glob(f"{stem}.*"), None)
        if img_path:
            break
    if not img_path:
        img_path = next((DATASET_DIR / "train" / "images").glob(f"{stem}.*"), None)
    if not img_path:
        continue
    results = best.predict(str(img_path), imgsz=IMG_SIZE, conf=PREDICT_CONF, verbose=False, device=DEVICE)
    pred = severity_for_prediction(results[0], id_to_name, cfg=CFG)
    y_true.append(row["severity"])
    y_pred.append(pred["severity"])

labels = ["none", "minor", "severe"]
acc = accuracy_score(y_true, y_pred)
print(f"\nSeverity accuracy: {acc:.3f}")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground truth (from GT boxes + thresholds)")
ax.set_title(f"Severity confusion matrix — accuracy {acc:.1%}")
plt.tight_layout()
plt.show()

## 10. Save or download the weights

The local app expects this file:

`runs/detect/ripeye/weights/best.pt`

Download it from Colab and place it at the same path in your local RipEye repo.

In [ ]:
assert WEIGHTS.exists(), f"Missing weights: {WEIGHTS}"
print("Trained weights:", WEIGHTS)
print("Size MB:", WEIGHTS.stat().st_size / 1024 / 1024)

# Option A: download directly from Colab
from google.colab import files
files.download(str(WEIGHTS))

## Optional: save weights to Google Drive

Run this if you want the weights to persist after the Colab runtime shuts down.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
out_dir = Path('/content/drive/MyDrive/RipEye')
out_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(WEIGHTS, out_dir / 'best.pt')
print('Saved to', out_dir / 'best.pt')